# **Notebook 5: Solution V1 — RAG Evaluation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] `./chroma_db/` — Created by Notebook 4
- [ ] `df_test.csv` — Created by Notebook 2
- [ ] `outputs.json` — Created by Notebooks 3+4
- [ ] GPU runtime enabled

**Files this notebook will CREATE:**
- [ ] `v1_metrics.csv` — Per-row Baseline vs V1 scores _(Evidence for comparative analysis)_

---

### **Task 3.3: Evaluate Solution V1**

> This task is split into five measurements (3.3.1–3.3.5). Run the shared setup cell below first (it loads the model, ChromaDB, and test data), then work through each measurement.

**── Shared setup ──**
Load the base model, reload ChromaDB (same embedding model as NB4), and load `df_test.csv` + `outputs.json`. Define helper functions `generate_baseline()` and `generate_naive_rag()` here so every subtask below can reuse them.

In [1]:
# YOUR CODE HERE

# Shared Setup for Notebook 5
# Load the model, tokenizer, vector database and test data.
# Define reusable functions for Baseline and Naive RAG generation.

import os
import json
import torch
import pandas as pd

from transformers import AutoTokenizer, AutoModelForCausalLM
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma


# 1. Load the Qwen model and tokenizer
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

print("Qwen model and tokenizer loaded successfully!")


# 2. Reload the existing ChromaDB vector database
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"}
)

vector_db = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings
)

print("ChromaDB loaded successfully!")
print("Total SOP documents:", vector_db._collection.count())


# 3. Load the held-out test dataset
df_test = pd.read_csv("df_test.csv")

print("Test dataset loaded:", len(df_test), "records")


# 4. Load outputs from Notebooks 3 and 4
with open("outputs.json", "r", encoding="utf-8") as f:
    baseline_data = json.load(f)

print("Previous outputs loaded successfully!")


# 5. Define a reusable function for Baseline generation
def generate_baseline(query):

    messages = [
        {"role": "user", "content": query}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return response.strip()


# 6. Define a reusable function for Naive RAG generation
def generate_naive_rag(query):

    retrieved_docs = vector_db.similarity_search(
        query,
        k=1
    )

    context = retrieved_docs[0].page_content

    system_prompt = f"""You are a customer support assistant.
Answer strictly using this SOP:

{context}

If the SOP does not contain the answer, say that the information is not available.
Do not invent information."""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": query}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return response.strip()


print("Baseline and Naive RAG functions defined successfully!")


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
W1119 19:04:08.646000 2744 .venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen model and tokenizer loaded successfully!


W1119 19:04:13.606000 2744 .venv\Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

ChromaDB loaded successfully!
Total SOP documents: 13
Test dataset loaded: 391 records
Previous outputs loaded successfully!
Baseline and Naive RAG functions defined successfully!



##### Shared Setup Conclusion

- The Qwen2.5-1.5B-Instruct model and tokenizer were loaded successfully.
- The existing ChromaDB vector database was reloaded with the same `all-MiniLM-L6-v2` embedding model used in Notebook 4.
- All 13 SOP documents were available in the vector database.
- The held-out test dataset was loaded successfully with 391 records.
- The previous results stored in `outputs.json` were loaded.
- Two reusable functions, `generate_baseline()` and `generate_naive_rag()`, were defined for generating responses using the baseline and Naive RAG approaches.

The shared setup is complete. The notebook is now ready to evaluate both approaches on the held-out test dataset.


#### **3.3.1 Execute Automated Testing [3 marks]**
**The Task:** Run both Baseline and Naive RAG across the entire held-out test set, collecting their generated outputs for every row.

**Hints & Tips:**
* Loop over `df_test` rows; for each query call both `generate_baseline()` and `generate_naive_rag()`.
* Store raw outputs in a list of dicts so the later measurements can score them.
* This is the most time-consuming cell — if constrained, `df_test.sample(50)` is acceptable.

**Learner Inference:** Automated testing across the full set gives statistically meaningful results, not a single cherry-picked query.

In [2]:
  # YOUR CODE HERE

# Task 3.3.1: Execute Automated Testing
# Run Baseline and Naive RAG on every held-out test query.
# Store both generated responses for evaluation in later tasks.

evaluation_results = []

# Loop through all records in the test dataset
for index, row in df_test.iterrows():

    # Get the customer query
    query = row["instruction"]

    # Generate response without retrieval
    baseline_response = generate_baseline(query)

    # Generate response using retrieved SOP context
    rag_response = generate_naive_rag(query)

    # Store query, expected labels and both generated responses
    evaluation_results.append({
        "instruction": query,
        "intent": row["intent"],
        "category": row["category"],
        "baseline_output": baseline_response,
        "naive_rag_output": rag_response
    })

    # Display progress after every 50 records
    if (index + 1) % 50 == 0:
        print(f"Processed {index + 1} records")

print("\nAutomated testing completed!")
print("Total records evaluated:", len(evaluation_results))


Processed 50 records
Processed 100 records
Processed 150 records
Processed 200 records
Processed 250 records
Processed 300 records
Processed 350 records

Automated testing completed!
Total records evaluated: 391



##### 3.3.1 Execute Automated Testing Conclusion

- Automated testing was completed successfully across the entire held-out test dataset.
- A total of 391 test records were evaluated.
- For each customer query, both the baseline model and Naive RAG generated responses.
- The generated outputs were stored for further evaluation and comparison.

Testing the complete held-out dataset provides a broader basis for comparing the baseline model with Naive RAG rather than relying on a single example.


In [3]:

# Inspect one evaluation record to identify the stored
# baseline and Naive RAG output fields.

print("Total evaluation records:", len(evaluation_results))
print("\nFirst record:")
print(evaluation_results[0])


Total evaluation records: 391

First record:
{'instruction': 'need help to shop several of ur item', 'intent': 'place_order', 'category': 'ORDER', 'baseline_output': "I'm sorry, but as an AI language model, I don't have any specific items or products that need assistance with shopping. However, if you could provide more details about the items you're interested in, I can try my best to assist you with your shopping needs.", 'naive_rag_output': "I'm sorry, but I don't have access to specific product details or inventory levels. However, if you need assistance finding multiple products, I suggest visiting our website's search function or contacting our customer service team who can guide you through the process."}


#### **3.3.2 Measure Format Adherence [2 marks]**
**The Task:** Validate the syntactic correctness of the generated outputs and report the adherence rate.

**Hints & Tips:**
* For the baseline/RAG free-text responses, "format adherence" means the output is well-formed and non-empty (the strict JSON check applies mainly to the fine-tuned router in NB7).
* Report the percentage of outputs that parsed/validated successfully.

**Learner Inference:** Format adherence tells you how often the system produces usable output before you even check correctness.

In [4]:
# YOUR CODE HERE

# Measure format adherence for Baseline and Naive RAG.
# A response is considered valid if it is a non-empty string.

total_records = len(evaluation_results)

baseline_valid = 0
naive_rag_valid = 0

for record in evaluation_results:

    # Check whether the Baseline response is non-empty.
    if isinstance(record["baseline_output"], str) and record["baseline_output"].strip():
        baseline_valid += 1

    # Check whether the Naive RAG response is non-empty.
    if isinstance(record["naive_rag_output"], str) and record["naive_rag_output"].strip():
        naive_rag_valid += 1

# Calculate format adherence percentages.
baseline_adherence = (baseline_valid / total_records) * 100
naive_rag_adherence = (naive_rag_valid / total_records) * 100

# Display the results.
print("Total records:", total_records)

print("\nBaseline Format Adherence:")
print("Valid responses:", baseline_valid)
print(f"Adherence rate: {baseline_adherence:.2f}%")

print("\nNaive RAG Format Adherence:")
print("Valid responses:", naive_rag_valid)
print(f"Adherence rate: {naive_rag_adherence:.2f}%")


Total records: 391

Baseline Format Adherence:
Valid responses: 391
Adherence rate: 100.00%

Naive RAG Format Adherence:
Valid responses: 391
Adherence rate: 100.00%



##### 3.3.2 Measure Format Adherence Conclusion

- Total test records evaluated: 391.
- Baseline model:
  - Valid responses: 391
  - Format adherence: 100.00%
- Naive RAG model:
  - Valid responses: 391
  - Format adherence: 100.00%

Both models generated non-empty responses for all 391 test records. This indicates that both systems consistently produced usable text outputs.

However, format adherence only measures whether a response is available and properly formed. It does not guarantee that the response is factually correct or follows the relevant SOP.

Therefore, further evaluation using ROUGE and BLEU scores is required to assess how closely the generated responses match SOP-grounded reference answers.


#### **3.3.3 Measure Execution Success (ROUGE/BLEU) [2 marks]**
**The Task:** Evaluate semantic similarity of each output against SOP-grounded references using ROUGE-1, ROUGE-L, and BLEU.

**Hints & Tips:**
* Use SOP-grounded references — retrieve the correct SOP per test row so policy-specific language is rewarded.
* Generic references falsely reward vague baseline answers — avoid them.
* `rouge_scorer.RougeScorer(['rouge1','rougeL'], use_stemmer=True)` and `sentence_bleu` with `SmoothingFunction().method1`.

**Learner Inference:** ROUGE/BLEU measure how close the output is to a correct, policy-grounded answer.

In [17]:

# Task 3.3.3: Measure Execution Success using ROUGE and BLEU
# Compare Baseline and Naive RAG against the relevant SOP documents.

from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import numpy as np

# Map each customer intent to its relevant SOP document.
# None indicates that no directly relevant SOP is available.

intent_to_sop = {
    "create_account": None,
    "delete_account": "data_privacy.md",
    "edit_account": "data_privacy.md",
    "recover_password": "account_recovery.md",
    "registration_problems": "account_recovery.md",
    "switch_account": "account_recovery.md",
    "check_cancellation_fee": "subscription_cancellation.md",
    "contact_customer_service": "working_hours.md",
    "contact_human_agent": "escalation_matrix.md",
    "delivery_options": None,
    "delivery_period": "shipping_delays.md",
    "complaint": "escalation_matrix.md",
    "review": None,
    "check_invoice": "billing_disputes.md",
    "get_invoice": "billing_disputes.md",
    "cancel_order": None,
    "change_order": "order_tracking.md",
    "place_order": None,
    "track_order": "order_tracking.md",
    "check_payment_methods": "payment_methods.md",
    "payment_issue": "payment_methods.md",
    "check_refund_policy": "refund_policy.md",
    "get_refund": "refund_policy.md",
    "track_refund": "refund_policy.md",
    "change_shipping_address": "order_tracking.md",
    "set_up_shipping_address": None,
    "newsletter_subscription": None
}

# Load the SOP documents from the existing ChromaDB database.
sop_data = vector_db.get(include=["documents", "metadatas"])

# Create a lookup using SOP filenames.
sop_lookup = {}

for content, metadata in zip(
    sop_data["documents"], sop_data["metadatas"]
):
    filename = metadata["source"].replace("\\", "/").split("/")[-1]
    sop_lookup[filename] = content

# Initialise ROUGE and BLEU scorers.
rouge = rouge_scorer.RougeScorer(
    ["rouge1", "rougeL"],
    use_stemmer=True
)

smoothing = SmoothingFunction().method1

baseline_scores = []
rag_scores = []

# Evaluate records that have a directly mapped SOP.
for record in evaluation_results:

    sop_filename = intent_to_sop.get(record["intent"])

    # Skip records without a directly mapped SOP.
    if sop_filename is None:
        continue

    reference = sop_lookup[sop_filename]
    reference_tokens = reference.split()

    # Evaluate Baseline output.
    baseline_output = record["baseline_output"]
    baseline_rouge = rouge.score(reference, baseline_output)

    baseline_bleu = sentence_bleu(
        [reference_tokens],
        baseline_output.split(),
        smoothing_function=smoothing
    )

    baseline_scores.append([
        baseline_rouge["rouge1"].fmeasure,
        baseline_rouge["rougeL"].fmeasure,
        baseline_bleu
    ])

    # Evaluate Naive RAG output.
    rag_output = record["naive_rag_output"]
    rag_rouge = rouge.score(reference, rag_output)

    rag_bleu = sentence_bleu(
        [reference_tokens],
        rag_output.split(),
        smoothing_function=smoothing
    )

    rag_scores.append([
        rag_rouge["rouge1"].fmeasure,
        rag_rouge["rougeL"].fmeasure,
        rag_bleu
    ])

# Calculate average scores.
baseline_avg = np.mean(baseline_scores, axis=0)
rag_avg = np.mean(rag_scores, axis=0)

# Display results.
print("Total test records:", len(evaluation_results))
print("Records evaluated with mapped SOPs:", len(baseline_scores))
print("Records without direct SOP mapping:",
      len(evaluation_results) - len(baseline_scores))

print("\nBASELINE AVERAGE SCORES")
print("ROUGE-1:", round(baseline_avg[0], 4))
print("ROUGE-L:", round(baseline_avg[1], 4))
print("BLEU:", round(baseline_avg[2], 4))

print("\nNAIVE RAG AVERAGE SCORES")
print("ROUGE-1:", round(rag_avg[0], 4))
print("ROUGE-L:", round(rag_avg[1], 4))
print("BLEU:", round(rag_avg[2], 4))


Total test records: 391
Records evaluated with mapped SOPs: 289
Records without direct SOP mapping: 102

BASELINE AVERAGE SCORES
ROUGE-1: 0.1791
ROUGE-L: 0.0839
BLEU: 0.002

NAIVE RAG AVERAGE SCORES
ROUGE-1: 0.1506
ROUGE-L: 0.0899
BLEU: 0.0044



##### 3.3.3 Measure Execution Success (ROUGE/BLEU) Conclusion

- A total of 391 test records were available for evaluation.
- Out of these, 289 records had directly mapped SOP documents and were used for ROUGE and BLEU evaluation.
- The remaining 102 records did not have direct SOP mappings and were excluded from the metric calculation.

**Baseline Scores:**
- ROUGE-1: 0.1791
- ROUGE-L: 0.0839
- BLEU: 0.0020

**Naive RAG Scores:**
- ROUGE-1: 0.1506
- ROUGE-L: 0.0899
- BLEU: 0.0044

- Baseline achieved a higher ROUGE-1 score.
- Naive RAG achieved slightly higher ROUGE-L and BLEU scores.
- Overall, Naive RAG did not consistently outperform the baseline across all three metrics.

The scores indicate limited text overlap between generated responses and the SOP references. ROUGE and BLEU measure lexical similarity and do not directly establish factual correctness or policy compliance.


#### **3.3.4 Measure Output Consistency [1 mark]**
**The Task:** Evaluate deterministic behaviour by running the same query multiple times under `do_sample=False` and confirming identical outputs.

**Hints & Tips:**
* Run the same query 3 times; assert all outputs are identical.
* With `do_sample=False, temperature=None`, greedy decoding should be fully deterministic.

**Learner Inference:** Deterministic inference means your evaluation is reproducible — the same input always gives the same output.

In [18]:
# YOUR CODE HERE

# Task 3.3.4: Measure Output Consistency

# Select one customer query from the test dataset.
test_query = df_test.iloc[0]["instruction"]

# Run the same query three times using Baseline.
baseline_runs = [
    generate_baseline(test_query)
    for _ in range(3)
]

# Run the same query three times using Naive RAG.
rag_runs = [
    generate_naive_rag(test_query)
    for _ in range(3)
]

# Check whether all three outputs are identical.
baseline_consistent = all(
    output == baseline_runs[0]
    for output in baseline_runs
)

rag_consistent = all(
    output == rag_runs[0]
    for output in rag_runs
)

# Display the results.
print("Customer Query:", test_query)

print("\nBaseline outputs identical:", baseline_consistent)
print("Naive RAG outputs identical:", rag_consistent)

print("\nBaseline consistency:", "PASS" if baseline_consistent else "FAIL")
print("Naive RAG consistency:", "PASS" if rag_consistent else "FAIL")

# Assert deterministic behaviour for both systems.
assert baseline_consistent, "Baseline outputs are not identical."
assert rag_consistent, "Naive RAG outputs are not identical."

print("\nOutput consistency test passed!")


Customer Query: need help to shop several of ur item

Baseline outputs identical: True
Naive RAG outputs identical: True

Baseline consistency: PASS
Naive RAG consistency: PASS

Output consistency test passed!



##### 3.3.4 Measure Output Consistency Conclusion

- The same customer query was executed three times using both Baseline and Naive RAG.
- Greedy decoding was enabled using `do_sample=False` in both generation functions.
- Baseline produced identical outputs across all three runs.
- Naive RAG also produced identical outputs across all three runs.
- Both systems passed the output consistency test.

This confirms that both systems produced reproducible outputs for the selected test query under the current inference settings.


#### **3.3.5 Measure Hallucination Frequency [2 marks]**
**The Task:** Evaluate how often outputs contain unsupported claims, invalid references, missing functionality, or policy violations.

**Hints & Tips:**
* Compare outputs against the retrieved SOP — flag any specific claim (dates, numbers, policies) not supported by the context.
* Report hallucination frequency as a percentage for both Baseline and Naive RAG.

**Learner Inference:** This quantifies the core problem RAG is meant to solve — grounding responses to reduce fabrication.

In [19]:
# YOUR CODE HERE

# Task 3.3.5: Measure Hallucination Frequency

import re

def get_terms(text):
    # Extract meaningful words from the text.
    words = re.findall(r"\b[a-zA-Z]{4,}\b", text.lower())
    return set(words)

baseline_hallucinations = 0
rag_hallucinations = 0
evaluated_records = 0

for record in evaluation_results:

    reference = record.get("reference", "")

    # Skip records without a directly mapped SOP.
    if reference.startswith("The available SOP documents"):
        continue

    reference_terms = get_terms(reference)

    baseline_terms = get_terms(record["baseline_output"])
    rag_terms = get_terms(record["naive_rag_output"])

    # Estimate unsupported content using word overlap.
    baseline_overlap = (
        len(baseline_terms & reference_terms) /
        max(len(baseline_terms), 1)
    )

    rag_overlap = (
        len(rag_terms & reference_terms) /
        max(len(rag_terms), 1)
    )

    # Flag answers with very low overlap for manual review.
    if baseline_overlap < 0.10:
        baseline_hallucinations += 1

    if rag_overlap < 0.10:
        rag_hallucinations += 1

    evaluated_records += 1


baseline_rate = baseline_hallucinations / evaluated_records * 100
rag_rate = rag_hallucinations / evaluated_records * 100

print("Total test records:", len(evaluation_results))
print("Records evaluated with mapped SOPs:", evaluated_records)

print("\nBASELINE")
print("Potential hallucinations:", baseline_hallucinations)
print(f"Potential hallucination rate: {baseline_rate:.2f}%")

print("\nNAIVE RAG")
print("Potential hallucinations:", rag_hallucinations)
print(f"Potential hallucination rate: {rag_rate:.2f}%")


Total test records: 391
Records evaluated with mapped SOPs: 289

BASELINE
Potential hallucinations: 46
Potential hallucination rate: 15.92%

NAIVE RAG
Potential hallucinations: 59
Potential hallucination rate: 20.42%



##### 3.3.5 Measure Hallucination Frequency Conclusion

The hallucination screening was performed on the 289 test records that had a directly mapped SOP reference. The remaining 102 records were excluded because no direct SOP mapping was available.

The baseline model produced 46 potential hallucinations, with a rate of 15.92%. Naive RAG produced 59 potential hallucinations, with a rate of 20.42%.

In this evaluation, Naive RAG showed a higher potential hallucination rate than the baseline. However, the measurement uses keyword overlap as a heuristic, so the flagged outputs should not automatically be considered actual hallucinations. Manual review would be needed to confirm unsupported claims.


### **Task 3.4: Analyse Retrieval Impact**

#### **3.4.1 Compare Baseline and Solution V1 [4 marks]**
**The Task:** Quantify the impact of retrieval by comparing aggregate scores across Functional Correctness, Consistency, and Hallucination Frequency, with percentage changes.

**Hints & Tips:**
* Build a summary table: Baseline vs Naive RAG for each metric.
* Compute improvement percentages: `(rag - base) / base * 100`.
* Document WHERE retrieval helps and where it doesn't — both motivate Stage 4.

**Learner Inference:** This isolates retrieval's independent contribution before fine-tuning enters the picture.

In [20]:
# YOUR CODE HERE

# Task 3.4.1: Compare Baseline and Solution V1

import pandas as pd

# Store the evaluation results from Task 3.3.
comparison = pd.DataFrame({
    "Metric": [
        "ROUGE-1",
        "ROUGE-L",
        "BLEU",
        "Potential Hallucination Rate (%)"
    ],
    "Baseline": [
        0.1791,
        0.0839,
        0.0020,
        15.92
    ],
    "Naive RAG": [
        0.1506,
        0.0899,
        0.0044,
        20.42
    ]
})

# Calculate percentage change.
comparison["Change (%)"] = (
    (comparison["Naive RAG"] - comparison["Baseline"])
    / comparison["Baseline"]
) * 100

print("BASELINE VS NAIVE RAG COMPARISON")
display(comparison.round(2))

print("\nOUTPUT CONSISTENCY")
print("Baseline: PASS")
print("Naive RAG: PASS")


BASELINE VS NAIVE RAG COMPARISON


,Metric,Baseline,Naive RAG,Change (%)
0,ROUGE-1,0.18,0.15,-15.91
1,ROUGE-L,0.08,0.09,7.15
2,BLEU,0.00,0.00,120.00
3,Potential Hallucination Rate (%),15.92,20.42,28.27



OUTPUT CONSISTENCY
Baseline: PASS
Naive RAG: PASS


---
## Save Artifacts

In [21]:
# YOUR CODE HERE

# Save Baseline vs Naive RAG comparison results

comparison.to_csv("v1_metrics.csv", index=False)

print("v1_metrics.csv saved successfully!")


v1_metrics.csv saved successfully!



## Notebook 5: Solution V1 (Naive RAG) Evaluation — Final Summary

In this notebook, the Baseline model and Solution V1 (Naive RAG) were evaluated using the held-out test dataset of 391 customer queries. The existing ChromaDB vector database was reloaded with 13 SOP documents.

### Evaluation Results

**1. Automated Testing**

Both Baseline and Naive RAG were executed across all 391 test records, and their generated responses were collected.

**2. Format Adherence**

Both systems achieved 100% format adherence, with 391 valid and non-empty responses each.

**3. ROUGE and BLEU Scores**

The evaluation was performed against SOP-grounded references for 289 records with a directly mapped SOP.

| Metric | Baseline | Naive RAG |
|---|---:|---:|
| ROUGE-1 | 0.1791 | 0.1506 |
| ROUGE-L | 0.0839 | 0.0899 |
| BLEU | 0.0020 | 0.0044 |

**4. Output Consistency**

Both Baseline and Naive RAG produced identical outputs across three runs of the same query. Both systems passed the consistency test.

**5. Hallucination Frequency**

A keyword-overlap heuristic was used to identify potential hallucinations among the 289 records with directly mapped SOPs.

- Baseline: 46 potential hallucinations (15.92%).
- Naive RAG: 59 potential hallucinations (20.42%).

These are heuristic estimates and require manual review to confirm actual unsupported claims.

**6. Retrieval Impact**

The comparison showed that Naive RAG improved ROUGE-L and BLEU scores but had a lower ROUGE-1 score than the Baseline. It also showed a higher potential hallucination rate under the keyword-overlap heuristic.

This indicates that retrieval did not improve every evaluation metric in the current implementation. The results highlight areas that can be addressed through further improvements in the next stage.

### Conclusion

Solution V1 successfully implemented retrieval using the SOP knowledge base and was evaluated against the Baseline model. Although some text-overlap metrics improved, the results were mixed, and the potential hallucination rate was higher for Naive RAG.

The evaluation results were saved in `v1_metrics.csv` for use in the final comparative analysis.


---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify before proceeding to Notebook 6.**

- [ ] ChromaDB reloaded from `./chroma_db/`
- [ ] **3.3.1** Automated testing run across full test set
- [ ] **3.3.2** Format adherence measured
- [ ] **3.3.3** ROUGE/BLEU computed with SOP-grounded references
- [ ] **3.3.4** Output consistency (determinism) verified
- [ ] **3.3.5** Hallucination frequency quantified
- [ ] **3.4.1** Retrieval impact quantified with improvement %
- [ ] **`v1_metrics.csv` saved** ← _Evidence for comparative analysis_

**If any item is unchecked, fix it before moving on.**


## END-OF-NOTEBOOK CHECKLIST — NOTEBOOK 5

- [x] ChromaDB reloaded successfully from `./chroma_db/`
- [x] 3.3.1 Automated testing completed across the full test set (391 records)
- [x] 3.3.2 Format adherence measured (100% for both systems)
- [x] 3.3.3 ROUGE-1, ROUGE-L and BLEU computed using SOP-grounded references (289 mapped records)
- [x] 3.3.4 Output consistency verified (both systems passed)
- [x] 3.3.5 Hallucination frequency quantified using keyword-overlap heuristic
- [x] 3.4.1 Baseline vs Naive RAG comparison completed with percentage changes
- [x] `v1_metrics.csv` saved successfully for comparative analysis

**All Notebook 5 tasks have been completed.**

Note: Hallucination percentages are heuristic estimates and have not been confirmed through manual review.
